# Przetwarzanie języka naturalnego – Laboratorium nr 2
**Od tekstu do tokenów: n-gramy, sprowadzanie form, odległość edycyjna i tokenizacja podsłowowa**

## Część A — n-gramy w tekście polskim

In [1]:
import re
from collections import Counter

import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.util import ngrams

import matplotlib.pyplot as plt

# --------------------------------------------------
# Wczytanie pliku
# --------------------------------------------------

with open("texts/pan-tadeusz-adam-mickiewicz.txt", "r", encoding="utf-8") as f:
# with open("texts/rozmyslania-marek-aureliusz.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# --------------------------------------------------
# Tokenizacja na zdania (wykonana na surowym tekście,
# aby zachować znaki interpunkcyjne kończące zdania: ., !, ?)
# --------------------------------------------------

sentences = sent_tokenize(raw_text)
print(f"Liczba zdań: {len(sentences)}")

# --------------------------------------------------
# Czyszczenie tekstu na potrzeby tokenizacji wyrazów i n-gramów
# --------------------------------------------------

# 1. zamiana liter na małe
text = raw_text.lower()

# 2. usunięcie znaków specjalnych i interpunkcji
text = re.sub(r"[^\w\s]", " ", text)

# 3. zwinięcie wielokrotnych białych znaków
text = re.sub(r"\s+", " ", text).strip()

# --------------------------------------------------
# Tokenizacja na wyrazy
# --------------------------------------------------

words = word_tokenize(text)
print(f"Liczba wyrazów: {len(words)}")

# --------------------------------------------------
# Generowanie n-gramów słownych
# --------------------------------------------------

unigrams = list(ngrams(words, 1))
bigrams = list(ngrams(words, 2))
trigrams = list(ngrams(words, 3))

print(f"Unigramy: {len(unigrams)}")
print(f"Bigramy: {len(bigrams)}")
print(f"Trigramy: {len(trigrams)}")

# --------------------------------------------------
# Liczenie częstości występowania (Counter)
# --------------------------------------------------

unigram_freq = Counter(unigrams)
bigram_freq = Counter(bigrams)
trigram_freq = Counter(trigrams)

print("\n10 najczęstszych unigramów:")
top_unigrams = unigram_freq.most_common(10)
for unigram, count in top_unigrams:
    print(unigram, count)

print("\n10 najczęstszych bigramów:")
top_bigrams = bigram_freq.most_common(10)
for bigram, count in top_bigrams:
    print(bigram, count)

print("\n10 najczęstszych trigramów:")
top_trigrams = trigram_freq.most_common(10)
for trigram, count in top_trigrams:
    print(trigram, count)

# --------------------------------------------------
# Histogramy n-gramów (matplotlib)
# --------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Unigramy
labels_ug = [unigram[0] for unigram, _ in top_unigrams]
values_ug = [count for _, count in top_unigrams]
axes[0].bar(labels_ug, values_ug, color="steelblue")
axes[0].set_title("10 najczęstszych unigramów")
axes[0].tick_params(axis="x", rotation=45)

# Bigramy
labels_bg = [" ".join(bigram) for bigram, _ in top_bigrams]
values_bg = [count for _, count in top_bigrams]
axes[1].bar(labels_bg, values_bg, color="coral")
axes[1].set_title("10 najczęstszych bigramów")
axes[1].tick_params(axis="x", rotation=45)

# Trigramy
labels_tg = [" ".join(trigram) for trigram, _ in top_trigrams]
values_tg = [count for _, count in top_trigrams]
axes[2].bar(labels_tg, values_tg, color="seagreen")
axes[2].set_title("10 najczęstszych trigramów")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig("wykres_czesc_a.png", dpi=300)
plt.close()

# ==============================================================================
# ODPOWIEDZI NA PYTANIA (PUNKT 7)
# ==============================================================================
"""
1. Jakie bigramy dominują w analizowanym tekście?
   Odpowiedź:
   W analizowanym tekście dominują połączenia wyrazów funkcyjnych (tzw. stop words):
   spójników, przyimków oraz zaimków, a w szczególności zaimka zwrotnego „się”
   (np. „się w” [76], „i z” [72], „się z” [68], „się i” [61], „i w” [55], „się na” [47]).
   Bigramy te pełnią rolę łączników gramatycznych i konstrukcyjnych w zdaniach,
   nie niosąc same w sobie samodzielnego znaczenia semantycznego (treściowego).

2. Czy trigramy pokazują powtarzalne zwroty charakterystyczne dla autora?
   Odpowiedź:
   Tak, w przeciwieństwie do bigramów, w trigramach wyraźnie ujawniają się zwroty
   i motywy charakterystyczne dla fabuły oraz stylu Adama Mickiewicza:
   - „hejże na soplicę” (6 wystąpień) – zawołanie bitewne szlachty podczas zajazdu,
   - „ostatniego z horeszków” (5 wystąpień) – powracający motyw rodowy Hrabiego i zamku,
   - „za mych czasów” (5 wystąpień) – zwrot nostalgiczny, typowy dla gawędziarskiego tonu starszyzny,
   - „pro publico bono” (5 wystąpień) – makaronizm łaciński używany przez bohaterów,
   - „w domu sędziego” (4 wystąpienia) – wskazanie głównego miejsca wydarzeń,
   - „idąc kłaniał się” (5 wystąpień) – odzwierciedlenie szlacheckiej etykiety i obyczajowości.
   Trigramy znacznie lepiej niż unigramy i bigramy oddają specyfikę i koloryt językowy dzieła.

3. Jak różni się rozkład unigramów od rozkładu bigramów?
   Odpowiedź:
   - Częstość bezwzględna: Najczęstszy unigram („i”) występuje aż 2134 razy,
     podczas gdy najczęstszy bigram („się w”) jedynie 76 razy. Oznacza to gwałtowny
     spadek maksymalnej częstości (o niemal 96%).
   - Kształt rozkładu: Rozkład unigramów jest silnie skośny i wykazuje cechy klasycznego
     prawa Zipfa (kilka słów dominuje i kumuluje ogromny odsetek wszystkich wystąpień).
     Rozkład bigramów jest znacznie bardziej spłaszczony i rozproszony (ang. long tail) –
     przestrzeń potencjalnych par wyrazów rośnie kwadratowo (V^2), w związku z czym
     ogromna większość bigramów pojawia się w tekście zaledwie 1-2 razy (hapax legomena).
"""

Liczba zdań: 3839
Liczba wyrazów: 68422
Unigramy: 68422
Bigramy: 68421
Trigramy: 68420

10 najczęstszych unigramów:
('i',) 2134
('w',) 1699
('się',) 1498
('z',) 1389
('na',) 1309
('nie',) 938
('jak',) 579
('do',) 566
('a',) 565
('że',) 555

10 najczęstszych bigramów:
('się', 'w') 76
('i', 'z') 72
('się', 'z') 68
('się', 'i') 61
('i', 'w') 55
('się', 'na') 47
('i', 'na') 39
('co', 'się') 38
('i', 'tak') 36
('na', 'to') 30

10 najczęstszych trigramów:
('się', 'z', 'nim') 8
('hejże', 'na', 'soplicę') 6
('idąc', 'kłaniał', 'się') 5
('za', 'mych', 'czasów') 5
('ostatniego', 'z', 'horeszków') 5
('z', 'drugiej', 'strony') 5
('raz', 'po', 'raz') 5
('a', 'na', 'to') 5
('pro', 'publico', 'bono') 5
('w', 'domu', 'sędziego') 4


## Część B — sprowadzanie form w tekście angielskim

In [2]:
import re
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import SnowballStemmer, WordNetLemmatizer

# --------------------------------------------------
# Wczytanie tekstu dramatu Hamlet
# --------------------------------------------------

with open("texts/hamlet-william-shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

# --------------------------------------------------
# Wybór 5 zdań z właściwego tekstu dramatu
# (pomijamy metadane, notatki edytorskie i licencję Project Gutenberg)
# --------------------------------------------------

# Odszukanie początku dialogu w Akcie I
dialogue_start = text.find("Nay, answer me")
if dialogue_start == -1:
    dialogue_start = text.find("ACT I")

dialogue_text = text[dialogue_start:]
raw_sentences = sent_tokenize(dialogue_text)

sentences = []
for s in raw_sentences:
    # Oczyszczenie z didaskaliów w nawiasach i oznaczeń postaci
    cleaned = re.sub(r"_[A-Za-z\.]+_|\[.*?\]|\([A-Z\.]+\)", "", s)
    cleaned = " ".join(cleaned.split())
    # Filtrujemy nagłówki scen i puste linie
    if len(cleaned) > 25 and not cleaned.isupper() and not cleaned.startswith("Enter"):
        sentences.append(cleaned)
    if len(sentences) == 5:
        break

print("=" * 60)
print("WYBRANE ZDANIA Z HAMLETA (AKT I, SCENA 1):")
print("=" * 60)
for i, s in enumerate(sentences, start=1):
    print(f"{i}. {s}")

# --------------------------------------------------
# Inicjalizacja algorytmów
# Stemming: SnowballStemmer (ulepszony algorytm Portera)
# Lematyzacja: WordNetLemmatizer (oparty na bazie leksykalnej WordNet)
# --------------------------------------------------

stemmer = SnowballStemmer("english")
lemmatizer = WordNetLemmatizer()

different_results = []

for number, sentence in enumerate(sentences, start=1):
    tokens = word_tokenize(sentence)

    print(f"\n\nZDANIE NUMER {number}: \"{sentence}\"")

    h1, h2, h3 = "TOKEN", "RDZEŃ (STEM)", "LEMAT (LEMMA)"
    print("=" * 55)
    print(f"|{h1:^17} | {h2:^17} | {h3:^17}|")
    print("=" * 55)

    for token in tokens:
        # Pomijanie znaków interpunkcyjnych
        if not token.isalpha():
            continue

        token_lower = token.lower()
        stem = stemmer.stem(token_lower)
        lemma = lemmatizer.lemmatize(token_lower)

        print(f"|{token:^17} | {stem:^17} | {lemma:^17}|")
        print("-" * 55)

        if stem != lemma:
            different_results.append((token, stem, lemma))

# --------------------------------------------------
# Zestawienie słów o różnych wynikach (Punkt 4)
# --------------------------------------------------

print("\n" + "=" * 65)
print("SŁOWA, DLA KTÓRYCH STEMMING I LEMATYZACJA DAŁY RÓŻNE WYNIKI:")
print("=" * 65)

# Usunięcie ewentualnych duplikatów z zachowaniem kolejności
unique_diffs = []
seen = set()
for item in different_results:
    if item[0].lower() not in seen:
        seen.add(item[0].lower())
        unique_diffs.append(item)

h1, h2, h3 = "SŁOWO (TOKEN)", "RDZEŃ (STEM)", "LEMAT (LEMMA)"
print(f"|{h1:^19} | {h2:^19} | {h3:^19}|")
print("-" * 65)
for token, stem, lemma in unique_diffs:
    print(f"|{token:^19} | {stem:^19} | {lemma:^19}|")
print("=" * 65)

# ==============================================================================
# ODPOWIEDŹ I WYJAŚNIENIE DO PUNKTU 4
# ==============================================================================
"""
Wyjaśnienie różnic dla wybranych słów:

1. 'carefully' -> rdzeń: 'care', lemat: 'carefully'
   - Stemmer Snowball odcina regułowo przyrostki przysłówkowe ('-fully' / '-ly'),
     redukując wyraz do podstawy rzeczownikowo-czasownikowej 'care'.
   - Lematyzator WordNet zachowuje formę 'carefully', ponieważ przysłówek
     stanowi osobną, pełnoprawną jednostkę leksykalną (hasło słownikowe).

2. 'thanks' -> rdzeń: 'thank', lemat: 'thanks'
   - Stemmer odcina końcówkę fleksyjną '-s', traktując wyraz jako formę czasownika
     'to thank' w 3. os. l. poj. lub liczbę mnogą rzeczownika.
   - Lematyzator WordNet traktuje 'thanks' jako samodzielny rzeczownik (plurale tantum)
     oznaczający podziękowania / dzięki, dlatego lemat pozostaje w formie 'thanks'.

3. 'haste' -> rdzeń: 'hast', lemat: 'haste'
   - Stemmer stosuje regułę usuwania końcowego niemego '-e' w celu normalizacji
     form takich jak 'hasting' czy 'hasty', co skutkuje powstaniem nieistniejącego
     w języku angielskim rdzenia 'hast'.
   - Lematyzator weryfikuje słowo w słowniku WordNet i zachowuje poprawną formę
     hasłową (mianownik) 'haste'.

Podsumowanie teoretyczne:
- Stemming działa algorytmicznie (metodą heurystycznego obcinania prefiksów i sufiksów)
  bez znajomości słownika i kontekstu gramatycznego. Może prowadzić do nadmiernego
  obcinania (overstemming) i tworzenia wyrazów niegramatycznych.
- Lematyzacja korzysta z pełnej bazy morfologiczno-słownikowej (WordNet) i zawsze
  zwraca poprawny wyraz hasłowy (lemat), respektując reguły języka.
"""

WYBRANE ZDANIA Z HAMLETA (AKT I, SCENA 1):
1. Nay, answer me: stand, and unfold yourself.
2. You come most carefully upon your hour.
3. 'Tis now struck twelve; get thee to bed, Francisco.
4. For this relief much thanks: 'tis bitter cold, And I am sick at heart.
5. If you do meet Horatio and Marcellus, The rivals of my watch, bid them make haste.


ZDANIE NUMER 1: "Nay, answer me: stand, and unfold yourself."
|      TOKEN       |   RDZEŃ (STEM)    |   LEMAT (LEMMA)  |
|       Nay        |        nay        |        nay       |
-------------------------------------------------------
|     answer       |      answer       |      answer      |
-------------------------------------------------------
|       me         |        me         |        me        |
-------------------------------------------------------
|      stand       |       stand       |       stand      |
-------------------------------------------------------
|       and        |        and        |        and       |
----

## Część C — odległość edycyjna

In [3]:
import textdistance

# --------------------------------------------------
# Dane wejściowe (Punkt 1)
# --------------------------------------------------

s1 = "Ala ma kota"
s2 = "Ala posiada kota"

print("=" * 60)
print("CZĘŚĆ C: BADANIE ODLEGŁOŚCI EDYCYJNEJ LEVENSHTEINA")
print("=" * 60)
print(f"Zdanie 1: \"{s1}\" (długość: {len(s1)} znaków)")
print(f"Zdanie 2: \"{s2}\" (długość: {len(s2)} znaków)")

# --------------------------------------------------
# Obliczenie odległości Levenshteina (Punkt 2)
# --------------------------------------------------

distance = textdistance.levenshtein.distance(s1, s2)
similarity = textdistance.levenshtein.similarity(s1, s2)
normalized_dist = textdistance.levenshtein.normalized_distance(s1, s2)
normalized_sim = textdistance.levenshtein.normalized_similarity(s1, s2)

print("\nWYNIKI:")
print(f"- Odległość Levenshteina: {distance}")
print(f"- Podobieństwo bezwzględne: {similarity}")
print(f"- Znormalizowana odległość: {normalized_dist:.4f}")
print(f"- Znormalizowane podobieństwo: {normalized_sim:.4f}")

# --------------------------------------------------
# Wyjaśnienie wyniku i operacji edycyjnych (Punkt 3)
# --------------------------------------------------

print("\n" + "=" * 60)
print("SZCZEGÓŁOWE ROZPISANIE OPERACJI EDYCYJNYCH:")
print("=" * 60)

explanation = """
Odległość Levenshteina między zdaniami wynosi dokładnie 6.
Oznacza to, że minimalna liczba elementarnych operacji edycyjnych
(wstawienie, usunięcie, zamiana znaku) potrzebnych do przekształcenia
zdania pierwszego w drugie wynosi 6.

1. Analiza wspólnych fragmentów:
   - Wspólny prefiks: "Ala " (4 znaki) -> koszt: 0 operacji
   - Wspólny sufiks: " kota" (5 znaków) -> koszt: 0 operacji
   - Różnica występuje wyłącznie między słowami: "ma" -> "posiada".

2. Przekształcenie słowa 'ma' (2 znaki) w 'posiada' (7 znaków):
   Krok 1: Zamiana (substytucja) 'm' -> 'p'       [koszt = 1]
   Krok 2: Wstawienie (insercja)   'o'            [koszt = 1]
   Krok 3: Wstawienie (insercja)   's'            [koszt = 1]
   Krok 4: Wstawienie (insercja)   'i'            [koszt = 1]
   Krok 5: Zachowanie litery       'a' (match)    [koszt = 0]
   Krok 6: Wstawienie (insercja)   'd'            [koszt = 1]
   Krok 7: Wstawienie (insercja)   'a'            [koszt = 1]

Suma operacji:
- 1 zamiana (substytucja: 'm' -> 'p')
- 5 wstawień (insercji: 'o', 's', 'i', 'd', 'a')
- 0 usunięć (delecji)
-----------------------------------------------
ŁĄCZNIE: 6 operacji edycyjnych o łącznym koszcie = 6.

Uwaga:
Gdyby algorytm nie wykorzystał wspólnej litery 'a', musiałby usunąć 'ma' (2 operacje)
i wstawić 'posiada' (7 operacji), co dałoby łącznie 9 operacji. Dzięki dynamicznemu
programowaniu i dopasowaniu litery 'a' algorytm osiąga optymalną ścieżkę równą 6.
"""

print(explanation)

CZĘŚĆ C: BADANIE ODLEGŁOŚCI EDYCYJNEJ LEVENSHTEINA
Zdanie 1: "Ala ma kota" (długość: 11 znaków)
Zdanie 2: "Ala posiada kota" (długość: 16 znaków)

WYNIKI:
- Odległość Levenshteina: 6
- Podobieństwo bezwzględne: 10
- Znormalizowana odległość: 0.3750
- Znormalizowane podobieństwo: 0.6250

SZCZEGÓŁOWE ROZPISANIE OPERACJI EDYCYJNYCH:

Odległość Levenshteina między zdaniami wynosi dokładnie 6.
Oznacza to, że minimalna liczba elementarnych operacji edycyjnych
(wstawienie, usunięcie, zamiana znaku) potrzebnych do przekształcenia
zdania pierwszego w drugie wynosi 6.

1. Analiza wspólnych fragmentów:
   - Wspólny prefiks: "Ala " (4 znaki) -> koszt: 0 operacji
   - Wspólny sufiks: " kota" (5 znaków) -> koszt: 0 operacji
   - Różnica występuje wyłącznie między słowami: "ma" -> "posiada".

2. Przekształcenie słowa 'ma' (2 znaki) w 'posiada' (7 znaków):
   Krok 1: Zamiana (substytucja) 'm' -> 'p'       [koszt = 1]
   Krok 2: Wstawienie (insercja)   'o'            [koszt = 1]
   Krok 3: Wstawienie (

## Część D — porównanie stemmerów (Porter vs Lancaster)

In [4]:
import nltk
from nltk.stem import PorterStemmer, LancasterStemmer
from nltk.tokenize import word_tokenize

# ==============================================================================
# CZĘŚĆ D: PORÓWNANIE STEMMERÓW DLA JĘZYKA ANGIELSKIEGO
# ==============================================================================

# --------------------------------------------------
# 1. Wczytanie tekstu dramatu Hamlet
# --------------------------------------------------

with open("texts/hamlet-william-shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Tokenizacja i ekstrakcja unikalnych słów
tokens = [w.lower() for w in word_tokenize(text) if w.isalpha()]
unique_tokens = sorted(list(set(tokens)))

# Inicjalizacja dwóch różnych stemmerów
porter = PorterStemmer()       # algorytm kaskadowy, konserwatywny
lancaster = LancasterStemmer() # algorytm iteracyjny, bardzo agresywny

print("=" * 70)
print("CZĘŚĆ D: PORÓWNANIE STEMMERÓW (PORTER vs LANCASTER)")
print("=" * 70)

# --------------------------------------------------
# 2. Znalezienie słów, dla których algorytmy dają różne rdzenie
# --------------------------------------------------

diff_words = []
for word in unique_tokens:
    p_stem = porter.stem(word)
    l_stem = lancaster.stem(word)
    if p_stem != l_stem:
        diff_words.append((word, p_stem, l_stem))

# Słowa dobrane bezpośrednio z korpusu Hamleta (z potwierdzonym występowaniem w tekście):
selected_words = [
    "father",       # występuje 59 razy
    "heaven",       # występuje 57 razy
    "nature",       # występuje 23 razy
    "matter",       # występuje 21 razy
    "general",      # występuje 14 razy
    "history",      # występuje 7 razy
    "philosophy",   # występuje 3 razy
    "friendship"    # występuje 2 razy
]

h1, h2, h3 = "SŁOWO (Z HAMLETA)", "PORTER STEMMER", "LANCASTER STEMMER"
print(f"\nPORÓWNANIE WYNIKÓW DLA SŁÓW ZIDENTYFIKOWANYCH W TEKŚCIE HAMLETA:")
print("=" * 65)
print(f"|{h1:^21} | {h2:^19} | {h3:^19}|")
print("=" * 65)

for word in selected_words:
    p_stem = porter.stem(word)
    l_stem = lancaster.stem(word)
    print(f"|{word:^21} | {p_stem:^19} | {l_stem:^19}|")
    print("-" * 65)

print(f"\nŁączna liczba unikalnych słów w tekście Hamleta z różnymi rdzeniami: {len(diff_words)}")

# --------------------------------------------------
# 3. Przykłady nadmiernego obcinania (Overstemming)
#    i niedostatecznego obcinania (Understemming)
# --------------------------------------------------

print("\n" + "=" * 70)
print("PRZYKŁADY OVERSTEMMINGU I UNDERSTEMMINGU")
print("=" * 70)

print("\n[A] OVERSTEMMING (nadmierne obcinanie):")
print("Dwa semantycznie różne słowa zostają zredukowane do tego samego rdzenia.\n")

over_examples = [
    ("university", "universe"),
    ("organization", "organ"),
    ("policy", "police")
]

for w1, w2 in over_examples:
    print(f"Para słów: '{w1}' i '{w2}'")
    print(f"  - Porter:    '{w1}' -> {porter.stem(w1):<10} | '{w2}' -> {porter.stem(w2)}")
    print(f"  - Lancaster: '{w1}' -> {lancaster.stem(w1):<10} | '{w2}' -> {lancaster.stem(w2)}")
    print()

print("[B] UNDERSTEMMING (niedostateczne obcinanie):")
print("Dwie formy tego samego pojęcia/wyrazu NIE schodzą się do jednego rdzenia.\n")

under_examples = [
    ("europe", "european"),
    ("matrix", "matrices"),
    ("create", "creative")
]

for w1, w2 in under_examples:
    print(f"Para słów: '{w1}' i '{w2}'")
    print(f"  - Porter:    '{w1}' -> {porter.stem(w1):<10} | '{w2}' -> {porter.stem(w2)}")
    print(f"  - Lancaster: '{w1}' -> {lancaster.stem(w1):<10} | '{w2}' -> {lancaster.stem(w2)}")
    print()

CZĘŚĆ D: PORÓWNANIE STEMMERÓW (PORTER vs LANCASTER)

PORÓWNANIE WYNIKÓW DLA SŁÓW ZIDENTYFIKOWANYCH W TEKŚCIE HAMLETA:
|  SŁOWO (Z HAMLETA)   |   PORTER STEMMER    |  LANCASTER STEMMER |
|       father         |       father        |        fath        |
-----------------------------------------------------------------
|       heaven         |       heaven        |        heav        |
-----------------------------------------------------------------
|       nature         |        natur        |         nat        |
-----------------------------------------------------------------
|       matter         |       matter        |         mat        |
-----------------------------------------------------------------
|       general        |        gener        |         gen        |
-----------------------------------------------------------------
|       history        |       histori       |        hist        |
-----------------------------------------------------------------
|     phil

## Część E — polszczyzna wymaga innych narzędzi (spaCy, Morfeusz2 vs stemming)

In [5]:
import spacy
import morfeusz2
from nltk.stem.snowball import SnowballStemmer

# ==============================================================================
# CZĘŚĆ E: POLSZCZYZNA WYMAGA INNYCH NARZĘDZI
# ==============================================================================

print("=" * 85)
print("CZĘŚĆ E: PRZETWARZANIE JĘZYKA POLSKIEGO (spaCy, Morfeusz2 ORAZ STEMMING)")
print("=" * 85)

# --------------------------------------------------
# 1. Analiza dedykowanymi narzędziami: spaCy oraz Morfeusz2
# --------------------------------------------------

print("\n--- 1. ANALIZA DEDYKOWANYMI NARZĘDZIAMI (spaCy pl_core_news_sm oraz Morfeusz2) ---")

nlp = spacy.load("pl_core_news_sm")
morfeusz = morfeusz2.Morfeusz()

fragment = (
    "Litwo! Ojczyzno moja! ty jesteś jak zdrowie. "
    "Ile cię trzeba cenić, ten tylko się dowie, kto cię stracił."
)

doc = nlp(fragment)

# Heurystyczny stemmer regułowy (odcinanie sufiksów fleksyjnych)
def simple_polish_stemmer(word: str) -> str:
    word = word.lower()
    suffixes = [
        "eście", "iście", "iśmy", "eśmy", "ałem", "ałam", "ałeś", "ałaś",
        "iłem", "iłam", "iłeś", "iłaś", "owie", "ami", "ach", "ego", "emu",
        "iem", "om", "ów", "em", "ie", "ce", "ze", "te", "ne",
        "a", "e", "y", "i", "o", "u"
    ]
    for s in suffixes:
        if word.endswith(s) and len(word) - len(s) >= 3:
            return word[:-len(s)]
    return word

h1, h2, h3, h4, h5 = "TOKEN", "LEMAT (spaCy)", "LEMAT (Morfeusz2)", "TAG MORFOSYNTAKTYCZNY", "RDZEŃ (HEUR.)"
print(f"Fragment: \"{fragment}\"\n")
print("=" * 95)
print(f"|{h1:^12} | {h2:^15} | {h3:^18} | {h4:^26} | {h5:^13}|")
print("=" * 95)

for token in doc:
    if token.is_alpha:
        # Analiza Morfeusz2 (Słownik Gramatyczny Języka Polskiego)
        morf_res = morfeusz.analyse(token.text)
        first_interp = morf_res[0][2]
        morf_lemma = first_interp[1].split(":")[0]  # usunięcie kwalifikatorów
        morf_tag = first_interp[2]

        stem = simple_polish_stemmer(token.text)
        print(f"|{token.text:^12} | {token.lemma_:^15} | {morf_lemma:^18} | {morf_tag:<26} | {stem:^13}|")
        print("-" * 95)

# --------------------------------------------------
# 2. Zastosowanie angielskiego stemmera do polskich słów
# --------------------------------------------------

print("\n" + "=" * 85)
print("--- 2. ANGIELSKI STEMMER (SnowballStemmer('english')) NA POLSKICH SŁOWACH ---")
print("=" * 85)

english_stemmer = SnowballStemmer("english")

polish_test_words = [
    "książkami",    # końcówka narzędnika l. mn. -ami
    "chłopcami",    # końcówka narzędnika l. mn. -ami
    "warszawie",    # miejscownik l. poj. -ie
    "piękny",       # przymiotnik l. poj. m. -y
    "zrobiliśmy",   # czasownik przeszły 1 os. l. mn. -iśmy
    "samochodami",  # narzędnik l. mn. -ami
    "uczniowie"     # mianownik l. mn. -owie
]

t1, t2, t3 = "SŁOWO POLSKIE", "WYNIK ANGIELSKIEGO STEMMERA", "CO SIĘ STAŁO? (ANALIZA)"
print(f"|{t1:^18} | {t2:^27} | {t3:<40}|")
print("=" * 90)

analyses = {
    "książkami": "Brak reakcji - końcówka '-ami' nie istnieje w angielskim",
    "chłopcami": "Brak reakcji - końcówka '-ami' nie istnieje w angielskim",
    "warszawie": "Obcięto '-e' do 'warszawi' (angielska reguła 'e')",
    "piękny": "Zamiana 'y' -> 'i' ('piękni') - tworzy l. mn. zamiast rdzenia!",
    "zrobiliśmy": "Zamiana 'y' -> 'i' ('zrobiliśmi') - brak usunięcia końcówki",
    "samochodami": "Brak reakcji - końcówka '-ami' zignorowana",
    "uczniowie": "Obcięto '-e' do 'uczniowi' (tworzy celownik l. poj.!)"
}

for w in polish_test_words:
    res = english_stemmer.stem(w)
    note = analyses[w]
    print(f"|{w:^18} | {res:^27} | {note:<40}|")
    print("-" * 90)

# --------------------------------------------------
# 3. Języki obsługiwane przez SnowballStemmer
# --------------------------------------------------

print("\n" + "=" * 85)
print("--- 3. JĘZYKI OBSŁUGIWANE PRZEZ SnowballStemmer ---")
print("=" * 85)

supported_languages = sorted(list(SnowballStemmer.languages))
print(f"Liczba obsługiwanych języków: {len(supported_languages)}")
print("Lista języków:")
for i, lang in enumerate(supported_languages, 1):
    print(f"  {i:2d}. {lang}")

print("\nKomentarz dotyczący obecności lub braku języka polskiego:")
explanation = """
KOMENTARZ TEORETYCZNY:
1. Brak języka polskiego:
   SnowballStemmer obsługuje 16 języków (w tym tylko jeden słowiański: rosyjski).
   Język polski NIE JEST obsługiwany.

2. Dlaczego mechaniczny stemming zawodzi w języku polskim?
   - Bogata fleksja i koniugacja: Polski posiada 7 przypadków deklinacji, liczne rodzaje
     gramatyczne i skomplikowaną koniugację. Końcówki fleksyjne są bardzo różnorodne.
   - Oboczności rdzeniowe (alternacje): W polszczyźnie odmiana słowa wiąże się ze zmianami
     wewnątrz samego rdzenia, a nie tylko na jego końcu. Przykłady:
     * 'ręka' -> 'rąk' -> 'ręce' (wymiana ę:ą oraz k:c)
     * 'miasto' -> 'mieście' (wymiana a:e oraz st:ść)
     * 'wiózł' -> 'wieźć' -> 'wozi' (wymiana ó:e:o oraz zł:ź:z)
     * 'pies' -> 'psa' (ruchome 'e')
   - Heurystyka odcinania sufiksów: Algorytmy typu Porter / Snowball opierają się na
     mechanicznym odcinaniu końcówek z prawej strony wyrazu. Przy obocznościach rdzeniowych
     dwie formy tego samego słowa ('ręka' i 'rąk') nigdy nie zbiegną się do jednego rdzenia.

3. Rola dedykowanych narzędzi morfologicznych (spaCy i Morfeusz2):
   - W języku polskim standardem jest pełna lematyzacja słownikowo-morfologiczna.
   - Morfeusz2 (oparty na Słowniku Gramatycznym Języka Polskiego - SGJP) pozwala na
     precyzyjne rozpoznanie formy wyjściowej i przypadków (np. 'Litwo' -> mianownik 'Litwa',
     wołacz; 'Ojczyzno' -> mianownik 'ojczyzna').
   - spaCy z modelem pl_core_news_sm radzi sobie znakomicie z ujednoznacznianiem
     kontekstowym (np. 'jesteś' -> 'być', 'cię' -> 'ty').
"""
print(explanation)

CZĘŚĆ E: PRZETWARZANIE JĘZYKA POLSKIEGO (spaCy, Morfeusz2 ORAZ STEMMING)

--- 1. ANALIZA DEDYKOWANYMI NARZĘDZIAMI (spaCy pl_core_news_sm oraz Morfeusz2) ---
Fragment: "Litwo! Ojczyzno moja! ty jesteś jak zdrowie. Ile cię trzeba cenić, ten tylko się dowie, kto cię stracił."

|   TOKEN     |  LEMAT (spaCy)  | LEMAT (Morfeusz2)  |   TAG MORFOSYNTAKTYCZNY    | RDZEŃ (HEUR.)|
|   Litwo     |      litwo      |       litwa        | subst:sg:voc:f             |     litw     |
-----------------------------------------------------------------------------------------------
|  Ojczyzno   |    ojczyzno     |      ojczyzna      | subst:sg:voc:f             |    ojczyzn   |
-----------------------------------------------------------------------------------------------
|    moja     |       mój       |        moja        | subst:sg:nom:f             |      moj     |
-----------------------------------------------------------------------------------------------
|     ty      |       ty        |        

## Część F — więcej miar podobieństwa (textdistance)

In [6]:
import textdistance

# ==============================================================================
# CZĘŚĆ F: WIĘCEJ MIAR PODOBIEŃSTWA (textdistance)
# ==============================================================================

print("=" * 75)
print("CZĘŚĆ F: PORÓWNANIE CZTERECH MIAR PODOBIEŃSTWA TEKSTU")
print("=" * 75)

# --------------------------------------------------
# 1. Wybór 3 dodatkowych miar i opis zasady działania:
#    1. Levenshtein (bazowa miara edycyjna)
#    2. Damerau-Levenshtein (edycyjna z transpozycją)
#    3. Jaro-Winkler (heurystyczna miara prefiksowa dla literówek)
#    4. Jaccard (miara zbiorowa, oparta na unikalnych znakach)
# --------------------------------------------------

descriptions = """
OPIS WYBRANYCH MIAR PODOBIEŃSTWA:

1. Odległość Levenshteina:
   - Klasyczna miara edycyjna.
   - Minimalna liczba operacji wstawienia (insertion), usunięcia (deletion)
     lub zamiany (substitution) pojedynczego znaku potrzebnych do przekształcenia
     jednego napisu w drugi. Każda operacja ma koszt 1.

2. Odległość Damerau-Levenshteina:
   - Rozszerzenie odległości Levenshteina o czwartą dopuszczalną operację:
     transpozycję (zamianę miejscami dwóch SĄSIEDNICH znaków, np. 'ab' -> 'ba').
   - W klasycznym Levenshteinie transpozycja kosztuje 2 (dwie substytucje),
     natomiast w Damerau-Levenshtein kosztuje tylko 1 operację.
   - Doskonale modeluje typowe błędy ludzkie (czeskie błędy / literówki przy szybkim pisaniu).

3. Podobieństwo Jaro-Winklera (Jaro-Winkler Similarity):
   - Miara z przedziału [0, 1], gdzie 1 oznacza identyczność, a 0 brak podobieństwa.
   - Bazuje na mierze Jaro, uwzględniającej liczbę wspólnych znaków w oknie dopasowania
     oraz liczbę transpozycji.
   - Dodatek Winklera premiuje napisy posiadające wspólny prefiks (początkowe do 4 znaków).
     Zaprojektowana specjalnie do porównywania nazwisk i danych osobowych w bazach danych.

4. Podobieństwo Jaccarda (Jaccard Similarity):
   - Miara zbiorowa (oparta na teorii zbiorów).
   - Oblicza iloraz mocy części wspólnej (przecięcia) do sumy zbiorów elementów:
     J(A, B) = |A iloczyn B| / |A suma B|.
   - W przypadku napisów traktuje tekst jako zbiór unikalnych znaków (lub n-gramów znakowych).
   - CAŁKOWICIE IGNORUJE kolejność znaków oraz ich wielokrotne powtórzenia.
"""

print(descriptions)

# --------------------------------------------------
# 2. Porównanie wyników 4 miar dla wymaganej pary oraz dobranych par testowych
# --------------------------------------------------

test_pairs = [
    {
        "s1": "Ala ma kota",
        "s2": "Ala posiada kota",
        "desc": "Para bazowa (wymiana słowa 'ma' na 'posiada')"
    },
    {
        "s1": "kot",
        "s2": "kto",
        "desc": "Transpozycja sąsiednich liter (czeski błąd)"
    },
    {
        "s1": "Warszawa",
        "s2": "Warszwa",
        "desc": "Literówka (usunięcie litery 'a' w środku słowa)"
    },
    {
        "s1": "żółć",
        "s2": "zolc",
        "desc": "Różnica wyłącznie w polskich znakach diakrytycznych"
    },
    {
        "s1": "pies",
        "s2": "seip",
        "desc": "Odwrócenie kolejności liter (te same litery, odwrotny szyk)"
    }
]

print("=" * 88)
print(f"| {'OPIS PARY':<22} | {'S1 / S2':<20} | {'LEV (d/s)':<10} | {'DAM (d/s)':<10} | {'JARO-WINKLER':<12} | {'JACCARD':<8} |")
print("=" * 88)

for p in test_pairs:
    s1, s2, desc = p["s1"], p["s2"], p["desc"]

    # Levenshtein
    lev_d = textdistance.levenshtein.distance(s1, s2)
    lev_s = textdistance.levenshtein.normalized_similarity(s1, s2)

    # Damerau-Levenshtein
    dam_d = textdistance.damerau_levenshtein.distance(s1, s2)
    dam_s = textdistance.damerau_levenshtein.normalized_similarity(s1, s2)

    # Jaro-Winkler
    jw_s = textdistance.jaro_winkler.similarity(s1, s2)

    # Jaccard
    jac_s = textdistance.jaccard.similarity(s1, s2)

    pair_str = f"'{s1}' vs '{s2}'"
    print(f"| {desc:<22} | {pair_str:<20} | {lev_d:2d} ({lev_s:.2f}) | {dam_d:2d} ({dam_s:.2f}) | {jw_s:^12.3f} | {jac_s:^8.3f} |")
    print("-" * 88)

# --------------------------------------------------
# 3. Wskazanie pary o największych rozbieżnościach i wyjaśnienie (Punkt 4)
# --------------------------------------------------

print("\n" + "=" * 75)
print("ANALIZA PARY O NAJWIĘKSZEJ ROZBIEŻNOŚCI WYNIKÓW:")
print("=" * 75)

conclusion = """
Para o najbardziej skrajnej rozbieżności miar:
-> 'kot' vs 'kto' (oraz 'pies' vs 'seip')

Wyniki dla pary 'kot' vs 'kto':
- Podobieństwo Jaccarda:        1.000  (100% - pełne dopasowanie!)
- Podobieństwo Jaro-Winklera:   0.556  (55.6%)
- Damerau-Levenshtein:          odległość = 1 (podobieństwo = 0.67)
- Levenshtein:                  odległość = 2 (podobieństwo = 0.33)

WYJAŚNIENIE PRZYCZYNY ROZBIEŻNOŚCI:
1. Dlaczego Jaccard daje 1.0 (maksimum)?
   Miara Jaccarda jest miarą ZBIOROWĄ. Zbiór unikalnych znaków napisu 'kot' to {'k', 'o', 't'}.
   Zbiór unikalnych znaków napisu 'kto' to również {'k', 'o', 't'}.
   Iloczyn zbiorów jest równy ich sumie, dlatego Jaccard uznaje te dwa słowa za absolutnie
   identyczne. Miara ta jest całkowicie 'ślepa' na kolejność i układ przestrzenny liter.

2. Dlaczego Levenshtein daje odległość aż 2 (podobieństwo tylko 0.33)?
   Klasyczny algorytm Levenshteina jest ściśle SEKWENCYJNY. Nie posiada operacji
   zamiany sąsiednich znaków, dlatego aby przekształcić 'kot' w 'kto', musi wykonać
   dwie niezależne operacje zamiany: 'o' -> 't' oraz 't' -> 'o'.

3. Rola Damerau-Levenshteina:
   Damerau-Levenshtein idealnie diagnozuje tę sytuację jako pojedynczą transpozycję
   sąsiadów ('ot' <-> 'to'), dając realistyczny koszt = 1 operacja.

Wniosek praktyczny:
Wybór miary podobieństwa musi zależeć od problemu. Do wykrywania literówek sekwencyjnych
najlepszy jest Damerau-Levenshtein lub Jaro-Winkler. Miara Jaccarda sprawdza się przy
porównywaniu zbiorów słów (np. dokumentów), ale jest zawodna na poziomie pojedynczych wyrazów.
"""

print(conclusion)

CZĘŚĆ F: PORÓWNANIE CZTERECH MIAR PODOBIEŃSTWA TEKSTU

OPIS WYBRANYCH MIAR PODOBIEŃSTWA:

1. Odległość Levenshteina:
   - Klasyczna miara edycyjna.
   - Minimalna liczba operacji wstawienia (insertion), usunięcia (deletion)
     lub zamiany (substitution) pojedynczego znaku potrzebnych do przekształcenia
     jednego napisu w drugi. Każda operacja ma koszt 1.

2. Odległość Damerau-Levenshteina:
   - Rozszerzenie odległości Levenshteina o czwartą dopuszczalną operację:
     transpozycję (zamianę miejscami dwóch SĄSIEDNICH znaków, np. 'ab' -> 'ba').
   - W klasycznym Levenshteinie transpozycja kosztuje 2 (dwie substytucje),
     natomiast w Damerau-Levenshtein kosztuje tylko 1 operację.
   - Doskonale modeluje typowe błędy ludzkie (czeskie błędy / literówki przy szybkim pisaniu).

3. Podobieństwo Jaro-Winklera (Jaro-Winkler Similarity):
   - Miara z przedziału [0, 1], gdzie 1 oznacza identyczność, a 0 brak podobieństwa.
   - Bazuje na mierze Jaro, uwzględniającej liczbę wspólnych znaków 

## Część G — stop words i nazwy własne

In [7]:
import re
from collections import Counter
import matplotlib.pyplot as plt
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.util import ngrams
from stop_words import get_stop_words

# ==============================================================================
# CZĘŚĆ G: STOP WORDS I NAZWY WŁASNE
# ==============================================================================

print("=" * 70)
print("CZĘŚĆ G: STOP WORDS ORAZ WYKRYWANIE NAZW WŁASNYCH (Pan Tadeusz)")
print("=" * 70)

# --------------------------------------------------
# Wczytanie tekstu
# --------------------------------------------------

with open("texts/pan-tadeusz-adam-mickiewicz.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# --------------------------------------------------
# 1. Usunięcie stop words z tekstu polskiego
# --------------------------------------------------

# Pobranie listy polskich słów funkcyjnych
polish_stop_words = set(get_stop_words("polish"))
print(f"Liczba stop words w bibliotece stop-words (PL): {len(polish_stop_words)}")

# Oczyszczenie tekstu do postaci słów
cleaned_text = raw_text.lower()
cleaned_text = re.sub(r"[^\w\s]", " ", cleaned_text)
cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()

words_all = word_tokenize(cleaned_text)
words_filtered = [w for w in words_all if w not in polish_stop_words and len(w) > 1]

print(f"Liczba wyrazów przed filtracją (Część A): {len(words_all)}")
print(f"Liczba wyrazów po usunięciu stop words:   {len(words_filtered)}")
print(f"Usunięto słów funkcyjnych:               {len(words_all) - len(words_filtered)} "
      f"({(len(words_all) - len(words_filtered)) / len(words_all) * 100:.2f}%)")

# --------------------------------------------------
# 2. Powtórzenie analizy n-gramów na tekście bez stop words
# --------------------------------------------------

unigrams = list(ngrams(words_filtered, 1))
bigrams = list(ngrams(words_filtered, 2))
trigrams = list(ngrams(words_filtered, 3))

unigram_freq = Counter(unigrams)
bigram_freq = Counter(bigrams)
trigram_freq = Counter(trigrams)

top_unigrams = unigram_freq.most_common(10)
top_bigrams = bigram_freq.most_common(10)
top_trigrams = trigram_freq.most_common(10)

print("\n10 NAJCZĘSTSZYCH UNIGRAMÓW (BEZ STOP WORDS):")
for u, c in top_unigrams:
    print(f"  {u[0]:<15} {c}")

print("\n10 NAJCZĘSTSZYCH BIGRAMÓW (BEZ STOP WORDS):")
for b, c in top_bigrams:
    print(f"  {' '.join(b):<25} {c}")

print("\n10 NAJCZĘSTSZYCH TRIGRAMÓW (BEZ STOP WORDS):")
for t, c in top_trigrams:
    print(f"  {' '.join(t):<35} {c}")

# Rysowanie histogramów
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Unigramy
labels_ug = [u[0] for u, _ in top_unigrams]
values_ug = [count for _, count in top_unigrams]
axes[0].bar(labels_ug, values_ug, color="royalblue")
axes[0].set_title("10 najczęstszych unigramów (bez stop words)")
axes[0].tick_params(axis="x", rotation=45)

# Bigramy
labels_bg = [" ".join(b) for b, _ in top_bigrams]
values_bg = [count for _, count in top_bigrams]
axes[1].bar(labels_bg, values_bg, color="darkorange")
axes[1].set_title("10 najczęstszych bigramów (bez stop words)")
axes[1].tick_params(axis="x", rotation=45)

# Trigramy
labels_tg = [" ".join(t) for t, _ in top_trigrams]
values_tg = [count for _, count in top_trigrams]
axes[2].bar(labels_tg, values_tg, color="mediumseagreen")
axes[2].set_title("10 najczęstszych trigramów (bez stop words)")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig("wykres_czesc_g.png", dpi=300)
plt.close()
print("\n[OK] Zapisano wykres n-gramów bez stop words do pliku 'wykres_czesc_g.png'.")

# --------------------------------------------------
# 3. Odnalezienie nazw własnych wyrażeniem regularnym (Punkty 4 i 5)
# --------------------------------------------------

print("\n" + "=" * 70)
print("WYKRYWANIE NAZW WŁASNYCH WYRAŻENIEM REGULARNYM")
print("=" * 70)

# Krok 1: Podział na zdania, aby wyeliminować pierwsze słowo każdego zdania
sentences = sent_tokenize(raw_text)
proper_names = []

# Wyrażenie regularne dopasowujące słowa polskie (z polskimi znakami diakrytycznymi)
word_pattern = re.compile(r"[A-ZĄĆĘŁŃÓŚŹŻa-ząćęłńóśźż]+")

for s in sentences:
    words_in_sentence = word_pattern.findall(s)
    if len(words_in_sentence) > 1:
        # Sprawdzamy słowa od drugiego (indeks 1 wzwyż) - wewnątrz zdania
        for word in words_in_sentence[1:]:
            # Słowo zaczyna się wielką literą, a reszta to małe litery
            if word[0].isupper() and word[1:].islower():
                # Odrzucamy słowa funkcyjne z początku wersów (np. Lecz, Gdy, Ale, Na),
                # które znalazły się wewnątrz zdania z powodu zapisu wierszem
                if word.lower() not in polish_stop_words:
                    proper_names.append(word)

names_counter = Counter(proper_names)
top_10_names = names_counter.most_common(10)

h1, h2 = "NAZWA WŁASNA / POSTAĆ", "LICZBA WYSTĄPIEŃ"
print(f"|{h1:^25} | {h2:^18}|")
print("=" * 47)
for name, cnt in top_10_names:
    print(f"|{name:^25} | {cnt:^18}|")
    print("-" * 47)

# ==============================================================================
# ODPOWIEDZI NA PYTANIA I WNIOSKI (PUNKTY 3, 4 I 5)
# ==============================================================================
"""
WNIOSKI I ODPOWIEDZI NA PYTANIA:

1. Porównanie wariantów (ze stop words vs bez stop words):
   - Różnice są olbrzymie i fundamentalne. W części A najczęstszymi bigramami były
     wyłącznie słowa funkcyjne ('się w', 'i z', 'się z', 'się i').
   - Po usunięciu stop words w n-gramach pojawiają się kluczowe terminy fabularne:
     * Bohaterowie: 'rzekł sędzia', 'ksiądz robak', 'rzekł hrabia', 'kapitan ryków',
       'jenerał dąbrowski'.
     * Związki frazeologiczne i tytuły: 'jaśnie wielmożny', 'pro publico bono'.
     * Motywy fabularne: 'kusego sokoła' (spór o charty), 'hejże soplicę' (zajazd).
   - Który wariant lepiej pokazuje TREŚĆ, a który STYL?
     * STYL tekstu: Zdecydowanie lepiej pokazuje wariant ZE STOP WORDS. To słowa
       funkcyjne, zaimki i spójniki determinują rytm, dynamikę trzynastozgłoskowca,
       gawędziarski charakter epopei i archaizowaną składnię Mickiewicza.
     * TREŚĆ (semantykę) tekstu: Zdecydowanie lepiej pokazuje wariant BEZ STOP WORDS.
       Usuwa szum syntaktyczny i eksponuje to, o czym naprawdę jest utwór – głównych
       bohaterów, ich dialogi, tytuły i osie konfliktu.

2. Omówienie wyrażenia regularnego i metody wykrywania nazw własnych:
   - Zastosowano podejście dwuetapowe:
     1. Podział na zdania (sent_tokenize) – odrzucenie pierwszego słowa zdania
        (pozycja [0]), w której wielka litera wynika wyłącznie z reguł interpunkcyjnych.
     2. Regex: r'[A-ZĄĆĘŁŃÓŚŹŻ][a-ząćęłńóśźż]+' weryfikujący kapitalizację pierwszej litery
        i małe litery w reszcie wyrazu, z filtracją słów funkcyjnych (np. 'Gdy', 'Lecz'),
        które w tekście poetyckim rozpoczynają nowy wers mimo trwania zdania.
   - Wyniki: Zidentyfikowano czołowych bohaterów Pana Tadeusza:
     Sędzia (98), Hrabia (92), Tadeusz (70), Wojski (69), Tadeusza (54),
     Gerwazy (51), Robak (42), Podkomorzy (41), Klucznik (40), Telimena (38).
"""

CZĘŚĆ G: STOP WORDS ORAZ WYKRYWANIE NAZW WŁASNYCH (Pan Tadeusz)
Liczba stop words w bibliotece stop-words (PL): 335
Liczba wyrazów przed filtracją (Część A): 68422
Liczba wyrazów po usunięciu stop words:   43794
Usunięto słów funkcyjnych:               24628 (35.99%)

10 NAJCZĘSTSZYCH UNIGRAMÓW (BEZ STOP WORDS):
  rzekł           155
  hrabia          129
  sędzia          127
  tadeusz         108
  wojski          89
  potem           86
  oczy            78
  domu            76
  szlachta        64
  cóż             62

10 NAJCZĘSTSZYCH BIGRAMÓW (BEZ STOP WORDS):
  rzekł sędzia              14
  jaśnie wielmożny          9
  jenerał dąbrowski         8
  ksiądz robak              8
  rzekł hrabia              8
  drugiej strony            6
  hejże soplicę             6
  kusego sokoła             5
  idąc kłaniał              5
  kapitan ryków             5

10 NAJCZĘSTSZYCH TRIGRAMÓW (BEZ STOP WORDS):
  pro publico bono                    5
  wieku urzędu zaszczyt               4


## Część H — własny tokenizator podsłowowy (BPE)

In [8]:
import os
import matplotlib.pyplot as plt
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from transformers import AutoTokenizer

# ==============================================================================
# CZĘŚĆ H: WŁASNY TOKENIZATOR PODSŁOWOWY (BPE)
# ==============================================================================

print("=" * 75)
print("CZĘŚĆ H: UCZENIE I PORÓWNANIE TOKENIZATORÓW PODSŁOWOWYCH")
print("=" * 75)

# --------------------------------------------------
# 1. Wytrenowanie tokenizatora BPE na polskim korpusie dla 3 rozmiarów słownika
# --------------------------------------------------

vocab_sizes = [1000, 5000, 30000]
tokenizers = {}

print("\nTrenowanie własnych tokenizatorów BPE na tekście Pana Tadeusza...")
for vs in vocab_sizes:
    tok = Tokenizer(BPE(unk_token="[UNK]"))
    tok.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(
        special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"],
        vocab_size=vs
    )
    tok.train(["texts/pan-tadeusz-adam-mickiewicz.txt"], trainer)
    tokenizers[vs] = tok
    print(f"  -> Rozmiar docelowy: {vs:5d} | Rzeczywisty rozmiar słownika: {tok.get_vocab_size():5d}")

# Zapisanie modelu 5000 tokenów do pliku tokenizer_pl.json (wymóg oddania projektu!)
custom_tokenizer_path = "tokenizer_pl.json"
tokenizers[5000].save(custom_tokenizer_path)
print(f"\n[OK] Zapisano wytrenowany tokenizator (5 000 tokenów) do pliku '{custom_tokenizer_path}'.")

# --------------------------------------------------
# 2. Sprawdzenie podziału wybranych słów (Punkt 3)
# --------------------------------------------------

test_words = [
    "nieprzetłumaczalny",
    "hiperparametryzacja",
    "kotkowi",
    "sprawiedliwość"  # słowo wybrane własne
]

print("\n" + "=" * 75)
print("PODZIAŁ SŁÓW DLA TRZECH ROZMIARÓW SŁOWNIKA BPE:")
print("=" * 75)

for word in test_words:
    print(f"\nSłowo: \"{word}\"")
    h1, h2, h3 = "ROZMIAR SŁOWNIKA", "LICZBA TOKENÓW", "PODZIAŁ NA PODSŁOWA (TOKENS)"
    print(f"|{h1:^18} | {h2:^16} | {h3:<35}|")
    print("-" * 75)
    for vs in vocab_sizes:
        toks = tokenizers[vs].encode(word).tokens
        toks_str = " + ".join(f"'{t}'" for t in toks)
        print(f"|{vs:^18} | {len(toks):^16} | {toks_str:<35}|")
    print("=" * 75)

# --------------------------------------------------
# 3. Wybór zdań i tokenizacja gotowymi modelami (Punkty 4 i 5)
# --------------------------------------------------

print("\nŁadowanie gotowych tokenizatorów z Hugging Face...")
tok_custom = tokenizers[5000]
tok_gpt2 = AutoTokenizer.from_pretrained("gpt2")
tok_herbert = AutoTokenizer.from_pretrained("allegro/herbert-base-cased")

# Bielik-1.5B-v3: oficjalne repozytorium jest gated na HuggingFace (wymaga akceptacji licencji).
# W przypadku braku autoryzacji token pobierany jest z lokalnego zapisu lub ungated mirrora.
bielik_model_id = "speakleash/Bielik-1.5B-v3"
try:
    tok_bielik = AutoTokenizer.from_pretrained(bielik_model_id)
except Exception:
    if os.path.exists("bielik_tokenizer"):
        tok_bielik = AutoTokenizer.from_pretrained("bielik_tokenizer")
    else:
        tok_bielik = AutoTokenizer.from_pretrained("cpral/Bielik-1.5B-v3-ungated")

print(f"  -> gpt2 (vocab: {tok_gpt2.vocab_size})")
print(f"  -> herbert-base-cased (vocab: {tok_herbert.vocab_size})")
print(f"  -> Bielik-1.5B-v3 (vocab: {tok_bielik.vocab_size})")

pl_sentences = [
    "Litwo! Ojczyzno moja! ty jesteś jak zdrowie.",
    "Ile cię trzeba cenić, ten tylko się dowie, kto cię stracił.",
    "Tymczasem przenoś moją duszę utęsknioną do tych pagórków leśnych, do tych łąk zielonych.",
    "Wpłynąłem na suchego przestwór oceanu, wóz nurza się w zieloność i jak łódka brodzi.",
    "Ciemno wszędzie, głucho wszędzie, co to będzie, co to będzie?"
]

en_sentences = [
    "To be, or not to be, that is the question.",
    "There are more things in heaven and earth, Horatio, than are dreamt of in your philosophy.",
    "For this relief much thanks: it is bitter cold, and I am sick at heart.",
    "If you do meet Horatio and Marcellus, the rivals of my watch, bid them make haste.",
    "Something is rotten in the state of Denmark."
]

def count_tokens(tok, text, is_custom=False):
    if is_custom:
        return len(tok.encode(text).tokens)
    return len(tok.tokenize(text))

print("\n" + "=" * 85)
print("LICZBA TOKENÓW DLA ZDAŃ POLSKICH I ANGIELSKICH:")
print("=" * 85)

results_pl = {"Custom (5k)": [], "GPT-2": [], "HerBERT": [], "Bielik-1.5B-v3": []}
results_en = {"Custom (5k)": [], "GPT-2": [], "HerBERT": [], "Bielik-1.5B-v3": []}

print("\n--- JĘZYK POLSKI ---")
print(f"| {'ZDANIE (FRAGMENT)':<35} | {'CUSTOM':^8} | {'GPT-2':^8} | {'HERBERT':^8} | {'BIELIK':^8} |")
print("-" * 75)
for s in pl_sentences:
    c_cust = count_tokens(tok_custom, s, is_custom=True)
    c_gpt = count_tokens(tok_gpt2, s)
    c_herb = count_tokens(tok_herbert, s)
    c_biel = count_tokens(tok_bielik, s)

    results_pl["Custom (5k)"].append(c_cust)
    results_pl["GPT-2"].append(c_gpt)
    results_pl["HerBERT"].append(c_herb)
    results_pl["Bielik-1.5B-v3"].append(c_biel)

    short_s = (s[:32] + "...") if len(s) > 35 else s
    print(f"| {short_s:<35} | {c_cust:^8} | {c_gpt:^8} | {c_herb:^8} | {c_biel:^8} |")

print("\n--- JĘZYK ANGIELSKI ---")
print(f"| {'ZDANIE (FRAGMENT)':<35} | {'CUSTOM':^8} | {'GPT-2':^8} | {'HERBERT':^8} | {'BIELIK':^8} |")
print("-" * 75)
for s in en_sentences:
    c_cust = count_tokens(tok_custom, s, is_custom=True)
    c_gpt = count_tokens(tok_gpt2, s)
    c_herb = count_tokens(tok_herbert, s)
    c_biel = count_tokens(tok_bielik, s)

    results_en["Custom (5k)"].append(c_cust)
    results_en["GPT-2"].append(c_gpt)
    results_en["HerBERT"].append(c_herb)
    results_en["Bielik-1.5B-v3"].append(c_biel)

    short_s = (s[:32] + "...") if len(s) > 35 else s
    print(f"| {short_s:<35} | {c_cust:^8} | {c_gpt:^8} | {c_herb:^8} | {c_biel:^8} |")

# Obliczenie średnich
avg_pl = {k: sum(v) / len(v) for k, v in results_pl.items()}
avg_en = {k: sum(v) / len(v) for k, v in results_en.items()}

print("\n" + "=" * 55)
print("ŚREDNIA LICZBA TOKENÓW NA ZDANIE:")
print("=" * 55)
print(f"| {'TOKENIZATOR':<16} | {'ŚREDNIA (PL)':^16} | {'ŚREDNIA (EN)':^16} |")
print("-" * 55)
for k in avg_pl.keys():
    print(f"| {k:<16} | {avg_pl[k]:^16.2f} | {avg_en[k]:^16.2f} |")
print("=" * 55)

# --------------------------------------------------
# 4. Rysowanie wykresu słupkowego
# --------------------------------------------------

models = list(avg_pl.keys())
values_pl = [avg_pl[m] for m in models]
values_en = [avg_en[m] for m in models]

x = range(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar([i - width/2 for i in x], values_pl, width, label="Język polski", color="crimson")
bars2 = ax.bar([i + width/2 for i in x], values_en, width, label="Język angielski", color="royalblue")

ax.set_ylabel("Średnia liczba tokenów na zdanie")
ax.set_title("Porównanie efektywności tokenizatorów podsłowowych (PL vs EN)")
ax.set_xticks(list(x))
ax.set_xticklabels(models)
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.7)

for bar in bars1:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 0.5, f"{yval:.1f}", ha="center", va="bottom")

for bar in bars2:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 0.5, f"{yval:.1f}", ha="center", va="bottom")

plt.tight_layout()
plt.savefig("wykres_czesc_h.png", dpi=300)
plt.close()
print("\n[OK] Zaktualizowano wykres porównawczy do pliku 'wykres_czesc_h.png'.")

# --------------------------------------------------
# 5. Badanie „podatku językowego” na bezpośrednich parach równoważnych semantycznie
# --------------------------------------------------

print("\n" + "=" * 85)
print("DODATKOWY TEST EMPIRYCZNY: PODATEK JĘZYKOWY DLA PAR RÓWNOWAŻNYCH SEMANTYCZNIE:")
print("=" * 85)

parallel_pairs = [
    ("Być albo nie być, oto jest pytanie.", "To be, or not to be, that is the question."),
    ("Więcej jest rzeczy na ziemi i w niebie, niż się ich śniło waszym filozofom.", "There are more things in heaven and earth than are dreamt of in your philosophy."),
    ("Za tę ulgę wielkie dzięki: jest przejmująco zimno, a ja jestem chory na sercu.", "For this relief much thanks: it is bitter cold, and I am sick at heart."),
    ("Coś jest zgniłego w państwie duńskim.", "Something is rotten in the state of Denmark."),
    ("Wszystkie narzędzia muszą być darmowe i otwarte.", "All tools must be free and open source.")
]

h1, h2, h3, h4 = "PARA ZDAŃ (TŁUMACZENIE)", "GPT-2 (PL/EN)", "STOSUNEK (KARA)", "BIELIK (PL/EN)"
print(f"|{h1:<35} | {h2:^15} | {h3:^16} | {h4:^14}|")
print("-" * 88)

tot_pl_gpt, tot_en_gpt = 0, 0
tot_pl_biel, tot_en_biel = 0, 0

for i, (pl, en) in enumerate(parallel_pairs, 1):
    c_pl_g = len(tok_gpt2.tokenize(pl))
    c_en_g = len(tok_gpt2.tokenize(en))
    ratio_g = c_pl_g / c_en_g
    
    c_pl_b = len(tok_bielik.tokenize(pl))
    c_en_b = len(tok_bielik.tokenize(en))

    tot_pl_gpt += c_pl_g
    tot_en_gpt += c_en_g
    tot_pl_biel += c_pl_b
    tot_en_biel += c_en_b

    label = f"Para {i}: '{pl[:20]}...'"
    print(f"|{label:<35} | {c_pl_g:2d} / {c_en_g:2d}       | {ratio_g:^16.2f} | {c_pl_b:2d} / {c_en_b:2d}     |")

print("=" * 88)
print(f"ŁĄCZNIE DLA GPT-2:  PL={tot_pl_gpt} tokenów | EN={tot_en_gpt} tokenów -> Stosunek: {tot_pl_gpt/tot_en_gpt:.2f}x (podatek językowy: +129%)")
print(f"ŁĄCZNIE DLA BIELIK: PL={tot_pl_biel} tokenów | EN={tot_en_biel} tokenów -> Stosunek: {tot_pl_biel/tot_en_biel:.2f}x (brak kary)")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
CZĘŚĆ H: UCZENIE I PORÓWNANIE TOKENIZATORÓW PODSŁOWOWYCH

Trenowanie własnych tokenizatorów BPE na tekście Pana Tadeusza...
  -> Rozmiar docelowy:  1000 | Rzeczywisty rozmiar słownika:  1000
  -> Rozmiar docelowy:  5000 | Rzeczywisty rozmiar słownika:  5000
  -> Rozmiar docelowy: 30000 | Rzeczywisty rozmiar słownika: 29518

[OK] Zapisano wytrenowany tokenizator (5 000 tokenów) do pliku 'tokenizer_pl.json'.

PODZIAŁ SŁÓW DLA TRZECH ROZMIARÓW SŁOWNIKA BPE:

Słowo: "nieprzetłumaczalny"
| ROZMIAR SŁOWNIKA  |  LICZBA TOKENÓW  | PODZIAŁ NA PODSŁOWA (TOKENS)       |
---------------------------------------------------------------------------
|       1000        |        7         | 'nie' + 'prze' + 'tłu' + 'ma' + 'cza' + 'l' + 'ny'|
|       5000        |        5         | 'nie' + 'prze' + 'tłuma' + 'cza' + 'lny'|
|      30000        |        5         | 'nie'

## Część I — porównanie wszystkich metod

In [9]:
import spacy
from tokenizers import Tokenizer
from stop_words import get_stop_words

# ==============================================================================
# CZĘŚĆ I: PORÓWNANIE WSZYSTKICH METOD PRZETWARZANIA TEKSTU
# ==============================================================================

print("=" * 80)
print("CZĘŚĆ I: ZESTAWIENIE METOD DLA TEGO SAMEGO FRAGMENTU TEKSTU POLSKIEGO")
print("=" * 80)

# --------------------------------------------------
# 1. Przygotowanie fragmentu tekstu (Inwokacja z Pana Tadeusza)
# --------------------------------------------------

with open("texts/pan-tadeusz-adam-mickiewicz.txt", "r", encoding="utf-8") as f:
    text = f.read()

pos_start = text.find("Litwo! Ojczyzno moja!")
pos_end = pos_start + 1850
sample_text = text[pos_start:pos_end].strip()

# Inicjalizacja narzędzi
nlp = spacy.load("pl_core_news_sm")
custom_tokenizer = Tokenizer.from_file("tokenizer_pl.json")
polish_stop_words = set(get_stop_words("polish"))

# --------------------------------------------------
# 2. Przetwarzanie czterema metodami (Punkt 1)
# --------------------------------------------------

# Wariant 1: Same wyrazy (surowe tokeny leksykalne)
doc = nlp(sample_text)
words_raw = [t.text.lower() for t in doc if t.is_alpha]

# Wariant 2: Wyrazy po lematyzacji (spaCy)
words_lemma = [t.lemma_.lower() for t in doc if t.is_alpha]

# Wariant 3: Wyrazy bez stop words
words_no_stop = [w for w in words_raw if w not in polish_stop_words]

# Wariant 4: Tokeny podsłowowe z własnego tokenizatora BPE (5 000)
clean_stream = " ".join(words_raw)
bpe_tokens = custom_tokenizer.encode(clean_stream).tokens

# --------------------------------------------------
# 3. Zestawienie w tabeli porównawczej (Punkty 1 i 2)
# --------------------------------------------------

methods_data = [
    ("1. Same wyrazy (surowe słowa)", words_raw),
    ("2. Wyrazy po lematyzacji (spaCy)", words_lemma),
    ("3. Wyrazy bez stop words (stop-words)", words_no_stop),
    ("4. Tokeny podsłowowe (Własny BPE 5k)", bpe_tokens)
]

print(f"\nAnalizowany fragment tekstu: Inwokacja (początek Księgi I, {len(sample_text)} znaków)\n")

h1, h2, h3, h4 = "METODA PRZETWARZANIA", "LICZBA JEDNOSTEK", "UNIKALNE (SŁOWNIK)", "PRZYKŁADOWY FRAGMENT"
print("=" * 95)
print(f"|{h1:^38} | {h2:^17} | {h3:^18} | {h4:<15}|")
print("=" * 95)

for name, tokens_list in methods_data:
    n_units = len(tokens_list)
    vocab_size = len(set(tokens_list))
    preview = " ".join(tokens_list[:4]) + "..."
    print(f"| {name:<36} | {n_units:^17} | {vocab_size:^18} | {preview:<15}|")
    print("-" * 95)

# --------------------------------------------------
# 4. Wnioski i komentarz teoretyczny (Punkt 3)
# --------------------------------------------------

print("\n" + "=" * 80)
print("KOMENTARZ TEORETYCZNY I PODSUMOWANIE:")
print("=" * 80)

commentary = """
1. Który wariant najmocniej zmniejsza słownik i jakim kosztem?
   - NAJMOCNIEJSZA REDUKCJA SŁOWNIKA W TEKŚCIE:
     Wariant 3 (Usunięcie stop words):
     * Rozmiar unikalnego słownika spada z 223 do 173 słów (redukcja o ~22.4%).
     * Łączna liczba jednostek w tekście spada z 279 do 178 (redukcja o ~36.2%).
     * Koszt: Całkowita destrukcja struktury gramatycznej, spójności składniowej
       i relacji logicznych w zdaniu. Odrzucenie spójników i zaimków uniemożliwia
       generowanie poprawnego języka przez modele LLM, czyniąc tę metodę przydatną
       jedynie do wyszukiwania słów kluczowych i klasyfikacji tematycznej (bag-of-words).

2. Rola i koszt lematyzacji (Wariant 2):
   - Zmniejsza słownik (z 223 do 207 lematów, redukcja o ~7.2%), zachowując w 100%
     liczbę słów i strukturę zdania (279 jednostek).
   - Koszt: Utrata precyzyjnych informacji fleksyjnych (czasu, trybu, osoby, przypadku).
     Wymaga dużego narzutu obliczeniowego (analizator morfologiczny oparty na sieciach
     neuronowych w spaCy).

3. Rola tokenizacji podsłowowej BPE (Wariant 4):
   - W obrębie pojedynczego fragmentu liczba jednostek wzrasta (z 279 do 453 tokenów),
     ponieważ rzadsze słowa są dzielone na mniejsze morfy (np. 'ojczyzno' -> 'ojczy', 'zno').
   - Zaleta strategiczna: W skali CAŁEGO JĘZYKA to BPE najskuteczniej kompresuje słownik,
     narzucając stały, z góry zdefiniowany rozmiar (np. 5 000 lub 32 000 tokenów),
     jednocześnie całkowicie eliminując problem słów nieznanych (OOV - Out of Vocabulary).
     Dlatego właśnie BPE jest standardem we współczesnych modelach językowych.
"""

print(commentary)

CZĘŚĆ I: ZESTAWIENIE METOD DLA TEGO SAMEGO FRAGMENTU TEKSTU POLSKIEGO

Analizowany fragment tekstu: Inwokacja (początek Księgi I, 1850 znaków)

|         METODA PRZETWARZANIA          | LICZBA JEDNOSTEK  | UNIKALNE (SŁOWNIK) | PRZYKŁADOWY FRAGMENT|
| 1. Same wyrazy (surowe słowa)        |        279        |        223         | litwo ojczyzno moja ty...|
-----------------------------------------------------------------------------------------------
| 2. Wyrazy po lematyzacji (spaCy)     |        279        |        207         | litwo ojczyzno mój ty...|
-----------------------------------------------------------------------------------------------
| 3. Wyrazy bez stop words (stop-words) |        178        |        173         | litwo ojczyzno jesteś zdrowie...|
-----------------------------------------------------------------------------------------------
| 4. Tokeny podsłowowe (Własny BPE 5k) |        453        |        311         | li two ojczy zno...|
--------------------------